# AutoEncoder : compression, espace latent et generation

In [ ]:
from src.dataset import load_mnist_dataset, load_shapes_npz
from src.autoencoder import AutoEncoder
from src.helper import extract_full_dataset, get_device
from src.metrics import compression_report, Latent

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

# Reproductibilite
np.random.seed(0)
torch.manual_seed(0)

print("device:", get_device())


EPOCHS = 15
EPOCHS_SWEEP = 6
BATCH_SIZE = 128

In [ ]:
def flattened_vector_to_image(flat_vector, image_shape):
    channels, height, width = image_shape
    image = flat_vector.reshape(channels, height, width)
    return image[0] if channels == 1 else np.transpose(image, (1, 2, 0))

def show_original_vs_reconstruction_grid(originals, reconstructions, image_shape, n=8, title=None):
    cmap = "gray" if image_shape[0] == 1 else None
    fig, axes = plt.subplots(2, n, figsize=(n * 1.3, 3))
    for i in range(n):
        axes[0, i].imshow(flattened_vector_to_image(originals[i], image_shape), cmap=cmap)
        axes[1, i].imshow(np.clip(flattened_vector_to_image(reconstructions[i], image_shape), 0, 1), cmap=cmap)
        for row in (0, 1):
            axes[row, i].set_xticks([]); axes[row, i].set_yticks([])
    axes[0, 0].set_ylabel("original"); axes[1, 0].set_ylabel("reconstruit")
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

def show_image_grid(flat_images, image_shape, nrow=4, ncol=8, title=None):
    cmap = "gray" if image_shape[0] == 1 else None
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.1, nrow * 1.1))
    for i, ax in enumerate(np.atleast_1d(axes).ravel()):
        ax.axis("off")
        if i < len(flat_images):
            ax.imshow(np.clip(flattened_vector_to_image(flat_images[i], image_shape), 0, 1), cmap=cmap)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

def plot_latent_scatter(latent_2d, labels, class_names=None, title=None):
    plt.figure(figsize=(6, 5))
    scatter = plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=labels, cmap="tab10", s=6, alpha=0.6)
    if class_names is not None:
        handles, _ = scatter.legend_elements()
        plt.legend(handles, class_names, title="classe", bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        plt.colorbar(scatter, label="chiffre")
    plt.xlabel("z1"); plt.ylabel("z2")
    if title:
        plt.title(title)
    plt.tight_layout(); plt.show()

def print_compression_report(report):
    for key, value in report.items():
        print(f"{key:>24}: {value:,.4f}" if isinstance(value, float) else f"{key:>24}: {value}")

def sample_gaussian_latent(latent_codes, n_samples, seed=0):
    rng = np.random.default_rng(seed)
    mean = latent_codes.mean(axis=0)
    cov = np.cov(latent_codes, rowvar=False)
    return rng.multivariate_normal(mean, cov, size=n_samples).astype(np.float32)

def interpolate_latent(z_start, z_end, steps=10):
    alphas = np.linspace(0, 1, steps)[:, None]
    return ((1 - alphas) * z_start[None, :] + alphas * z_end[None, :]).astype(np.float32)

def run_autoencoder_hyperparam_experiment(
        X_train, X_eval, input_dim, latent_dim, activation, epochs
        ):
    model = AutoEncoder(input_dim=input_dim, output_dim=input_dim, latent_dim=latent_dim,
                        encoder_layer_num=3, decoder_layer_num=3, encoder_activation=activation)
    model.fit(X_train, epochs=epochs, batch_size=BATCH_SIZE)
    latent = model.encode(X_eval)
    reconstruction = model.decode(latent)
    return compression_report(model.get_codebook(), latent, X_eval, reconstruction)

def subsample_dataset(images, labels, n, seed=0):
    if n >= len(images):
        return images, labels
    idx = np.random.default_rng(seed).choice(len(images), size=n, replace=False)
    return images[idx], labels[idx]

## Partie A - MNIST DIGITS

In [ ]:
mnist_train_images, mnist_train_labels = extract_full_dataset(load_mnist_dataset(train=True, shuffle=False))
mnist_eval_images, mnist_eval_labels = extract_full_dataset(load_mnist_dataset(train=False, shuffle=False))

MNIST_SHAPE = (1, 28, 28)
X_mnist_train, y_mnist_train = subsample_dataset(
    mnist_train_images.reshape(len(mnist_train_images), -1).numpy(), mnist_train_labels.numpy(), 15000
    )
X_mnist_eval, y_mnist_eval = subsample_dataset(
    mnist_eval_images.reshape(len(mnist_eval_images), -1).numpy(), mnist_eval_labels.numpy(), 3000
    )
print("train:", X_mnist_train.shape, "| eval:", X_mnist_eval.shape)

### Train - ReLu et MSE comme baseline

In [ ]:
mnist_model = AutoEncoder(
    input_dim=784, output_dim=784, latent_dim=2,
    encoder_layer_num=3, decoder_layer_num=3, encoder_activation=nn.ReLU, fonction_loss=nn.MSELoss
)
mnist_model.fit(X_mnist_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

plt.figure(figsize=(5, 3))
plt.plot(mnist_model.loss_history, marker="o")
plt.xlabel("epoch"); plt.ylabel("MSE d'entrainement"); plt.title("Courbe d'apprentissage MNIST")
plt.tight_layout(); plt.show()

### Compression et decompression

In [ ]:
mnist_latent = mnist_model.encode(X_mnist_eval)
mnist_reconstructed = mnist_model.decode(mnist_latent)
print("code latent par image:", mnist_latent.array.shape[1], "valeurs | nature:", mnist_latent.nature)
show_original_vs_reconstruction_grid(X_mnist_eval, mnist_reconstructed, MNIST_SHAPE, n=8,
                     title="MNIST: original (haut) vs reconstruit (bas)")

### Qualite de reconstruction et taille du message

In [ ]:
mnist_report = compression_report(mnist_model.get_codebook(), mnist_latent, X_mnist_eval, mnist_reconstructed)
print_compression_report(mnist_report)

per_image_bytes = mnist_latent.n_bytes / len(X_mnist_eval)
print(f"\nMessage transmis par image: {per_image_bytes:.0f} octets ({mnist_latent.array.shape[1]} float32),")
print(f"contre {X_mnist_eval[0].nbytes} octets pour l'image originale en float32.")
print(f"Codebook (poids du decodeur): {mnist_report['codebook_bytes']:,} octets, partage une seule fois.")

### Visualisation de l'espace latent

In [ ]:
plot_latent_scatter(mnist_latent, y_mnist_eval, title="Espace latent 2D des chiffres MNIST")

### Generation de nouvelles images

Le decodeur seul est un generateur: on lui fournit des codes latents inedits.

- Echantillonnage gaussien: on ajuste une gaussienne sur les codes du jeu d'evaluation et on en tire de nouveaux.
- Interpolation: on relie deux images reelles par une droite dans l'espace latent.
- Balayage de la variete 2D: on decode une grille reguliere du plan latent du modele 2D.

Note: un AutoEncoder classique ne contraint pas son espace latent a une loi connue; rester pres de la distribution des donnees (gaussienne ajustee, interpolation) donne les images les plus nettes.

In [ ]:
# Echantillonnage gaussien
new_codes = sample_gaussian_latent(mnist_latent.array, n_samples=32)
generated = mnist_model.decode(Latent(array=new_codes, nature="continuous"))
show_image_grid(generated, MNIST_SHAPE, nrow=4, ncol=8, title="Chiffres generes (echantillonnage gaussien)")

# Interpolation entre deux images
path = interpolate_latent(mnist_latent.array[0], mnist_latent.array[1], steps=10)
morph = mnist_model.decode(Latent(array=path, nature="continuous"))
show_image_grid(morph, MNIST_SHAPE, nrow=1, ncol=10, title="Interpolation dans l'espace latent")

# Balayage du latent
lo, hi = mnist_latent.min(axis=0), mnist_latent.max(axis=0)
grid_axis = 12
xs = np.linspace(lo[0], hi[0], grid_axis)
ys = np.linspace(hi[1], lo[1], grid_axis)
grid = np.array([[x, y] for y in ys for x in xs], dtype=np.float32)
manifold = mnist_model.decode(Latent(array=grid, nature="continuous"))
show_image_grid(manifold, MNIST_SHAPE, nrow=grid_axis, ncol=grid_axis, title="Variete generee par le plan latent 2D")

### Experimentation sur les hyper-parametres

In [ ]:
latent_dims = [2, 8, 16, 32, 64]
activations = {"ReLU": nn.ReLU, "Tanh": nn.Tanh}
X_tr, y_tr = subsample_dataset(X_mnist_train, y_mnist_train, 6000, seed=1)

mnist_results = []
for act_name, act in activations.items():
    for latent_dim in latent_dims:
        report = run_autoencoder_hyperparam_experiment(X_tr, X_mnist_eval, 784, latent_dim, act, EPOCHS_SWEEP)
        report.update(latent_dim=latent_dim, activation=act_name)
        mnist_results.append(report)
        print(f"{act_name:>5} | latent={latent_dim:>2} | MSE={report['reconstruction_mse']:.4f} "
              f"| total={report['total_compressed_bytes']:>9,} o | ratio={report['compression_ratio']:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for act_name in activations:
    pts = [r for r in mnist_results if r["activation"] == act_name]
    axes[0].plot([r["latent_dim"] for r in pts], [r["reconstruction_mse"] for r in pts], marker="o", label=act_name)
    axes[1].plot([r["total_compressed_bytes"] for r in pts], [r["reconstruction_mse"] for r in pts], marker="o", label=act_name)
axes[0].set_xlabel("dimension latente"); axes[0].set_ylabel("MSE"); axes[0].set_title("Qualite vs dimension latente"); axes[0].legend()
axes[1].set_xlabel("taille compressee totale (octets)"); axes[1].set_ylabel("MSE"); axes[1].set_title("Compromis taille / qualite"); axes[1].legend()
plt.tight_layout(); plt.show()

### Variation de la fonction de perte
MSE, L1 et BCE

In [ ]:
loss_functions = {"MSE": nn.MSELoss, "L1": nn.L1Loss, "BCE": nn.BCELoss}

mnist_loss_mse = {}
for name, loss_cls in loss_functions.items():
    model = AutoEncoder(input_dim=784, output_dim=784, latent_dim=32,
                        encoder_layer_num=3, decoder_layer_num=3,
                        encoder_activation=nn.ReLU, fonction_loss=loss_cls)
    model.fit(X_tr, epochs=EPOCHS_SWEEP, batch_size=BATCH_SIZE)
    latent = model.encode(X_mnist_eval)
    reconstruction = model.decode(latent)
    mnist_loss_mse[name] = compression_report(model.get_codebook(), latent, X_mnist_eval, reconstruction)["reconstruction_mse"]
    show_original_vs_reconstruction_grid(X_mnist_eval, reconstruction, MNIST_SHAPE, n=8,
                         title=f"MNIST reconstruit - perte {name} (MSE={mnist_loss_mse[name]:.4f})")

plt.figure(figsize=(4, 3))
plt.bar(list(mnist_loss_mse), list(mnist_loss_mse.values()))
plt.ylabel("MSE d'evaluation"); plt.title("MNIST: impact de la fonction de perte")
plt.tight_layout(); plt.show()

## Partie B - shapes

In [ ]:
X_shapes_train, y_shapes_train, shape_names = load_shapes_npz(split="train", max_samples=12000)
X_shapes_eval, y_shapes_eval, _ = load_shapes_npz(split="validation", max_samples=3000)
X_shapes_train = X_shapes_train.reshape(len(X_shapes_train), -1)
X_shapes_eval = X_shapes_eval.reshape(len(X_shapes_eval), -1)

SHAPES_SHAPE = (3, 32, 32)
SHAPES_DIM = 3 * 32 * 32
print("train:", X_shapes_train.shape, "| eval:", X_shapes_eval.shape, "| classes:", shape_names)
show_image_grid(X_shapes_eval[:8], SHAPES_SHAPE, nrow=1, ncol=8, title="Exemples de formes")

### Train, compression et decompression

In [ ]:
shapes_model = AutoEncoder(input_dim=SHAPES_DIM, output_dim=SHAPES_DIM, latent_dim=2,
                           encoder_layer_num=3, decoder_layer_num=3, encoder_activation=nn.ReLU)
shapes_model.fit(X_shapes_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

plt.figure(figsize=(5, 3))
plt.plot(shapes_model.loss_history, marker="o")
plt.xlabel("epoch"); plt.ylabel("MSE d'entrainement"); plt.title("Courbe d'apprentissage shapes")
plt.tight_layout(); plt.show()

shapes_latent = shapes_model.encode(X_shapes_eval)
shapes_reconstructed = shapes_model.decode(shapes_latent)
show_original_vs_reconstruction_grid(X_shapes_eval, shapes_reconstructed, SHAPES_SHAPE, n=8,
                     title="Shapes: original (haut) vs reconstruit (bas)")

### Metriques de compression

In [ ]:
shapes_report = compression_report(shapes_model.get_codebook(), shapes_latent, X_shapes_eval, shapes_reconstructed)
print_compression_report(shapes_report)

per_image_bytes = shapes_latent.n_bytes / len(X_shapes_eval)
print(f"\nMessage par image: {per_image_bytes:.0f} octets contre {X_shapes_eval[0].nbytes} octets en float32.")

### Espace latent et generation

In [ ]:
plot_latent_scatter(shapes_latent, y_shapes_eval, class_names=shape_names,
                    title="Espace latent 2D des formes")

# Generation par echantillonnage gaussien
new_codes = sample_gaussian_latent(shapes_latent.array, n_samples=16)
generated = shapes_model.decode(Latent(array=new_codes, nature="continuous"))
show_image_grid(generated, SHAPES_SHAPE, nrow=2, ncol=8, title="Formes generees (echantillonnage gaussien)")

# Interpolation entre deux formes
path = interpolate_latent(shapes_latent.array[0], shapes_latent.array[5], steps=10)
morph = shapes_model.decode(Latent(array=path, nature="continuous"))
show_image_grid(morph, SHAPES_SHAPE, nrow=1, ncol=10, title="Interpolation entre deux formes")

### Experimentation hyper-parametres

In [ ]:
X_shapes_tr, _ = subsample_dataset(X_shapes_train, y_shapes_train, 6000, seed=1)
shapes_results = []
for latent_dim in [2, 8, 16, 32, 64]:
    report = run_autoencoder_hyperparam_experiment(X_shapes_tr, X_shapes_eval, SHAPES_DIM, latent_dim, nn.ReLU, EPOCHS_SWEEP)
    report.update(latent_dim=latent_dim)
    shapes_results.append(report)
    print(f"latent={latent_dim:>2} | MSE={report['reconstruction_mse']:.4f} "
          f"| total={report['total_compressed_bytes']:>9,} o | ratio={report['compression_ratio']:.2f}")

plt.figure(figsize=(5, 4))
plt.plot([r["latent_dim"] for r in shapes_results], [r["reconstruction_mse"] for r in shapes_results], marker="o")
plt.xlabel("dimension latente"); plt.ylabel("MSE"); plt.title("Shapes: qualite vs dimension latente")
plt.tight_layout(); plt.show()

### Variation de la fonction de perte
MSE, L1, BCE

In [ ]:
shapes_loss_mse = {}
for name, loss_cls in loss_functions.items():
    model = AutoEncoder(input_dim=SHAPES_DIM, output_dim=SHAPES_DIM, latent_dim=2,
                        encoder_layer_num=3, decoder_layer_num=3,
                        encoder_activation=nn.ReLU, fonction_loss=loss_cls)
    model.fit(X_shapes_tr, epochs=EPOCHS_SWEEP, batch_size=BATCH_SIZE)
    latent = model.encode(X_shapes_eval)
    reconstruction = model.decode(latent)
    shapes_loss_mse[name] = compression_report(model.get_codebook(), latent, X_shapes_eval, reconstruction)["reconstruction_mse"]
    show_original_vs_reconstruction_grid(X_shapes_eval, reconstruction, SHAPES_SHAPE, n=8,
                         title=f"Shapes reconstruit - perte {name} (MSE={shapes_loss_mse[name]:.4f})")

plt.figure(figsize=(4, 3))
plt.bar(list(shapes_loss_mse), list(shapes_loss_mse.values()))
plt.ylabel("MSE d'evaluation"); plt.title("Shapes: impact de la fonction de perte")
plt.tight_layout(); plt.show()

## Reponses aux questions du projet

1. Nature de l'espace latent.

Il est continu: encode renvoie un Latent de nature "continuous", c'est-a-dire un vecteur de reels (float32) dense. N'importe quel point de l'espace peut etre decode, ce qui rend possibles l'interpolation et l'echantillonnage. Cela contraste avec K-Means, dont l'espace latent est discret (un indice de cluster parmi K).

2. Codebook.

Le codebook renvoye par get_codebook regroupe les poids du decodeur. C'est le dictionnaire partage: l'emetteur et le recepteur doivent tous deux en disposer pour reconstruire. Concretement, l'emetteur compresse avec l'encodeur et n'envoie que le code latent; le recepteur applique le decodeur (le codebook) a ce code. Le codebook est un cout fixe, paye une seule fois et amorti sur toutes les images.

3. Qualite de reconstruction.

Mesuree par la MSE de compression_report (voir les tableaux ci-dessus) et visible sur les grilles original vs reconstruit. Elle s'ameliore quand la dimension latente augmente, au prix d'un message plus gros: c'est le compromis illustre par les courbes d'experimentation. Elle depend aussi de la fonction de perte d'entrainement (MSE, L1, BCE).

4. Code byte (taille du message).

Le message transmis par image est le code latent: dimension_latente x 4 octets (float32), soit par exemple 128 octets pour une dimension latente de 32. La taille totale pour un lot est latent_bytes dans compression_report; le codebook s'y ajoute une seule fois, independamment du nombre d'images.